# 🎙️ VoiceBatch Studio v2.1.4 - [Master Final]
इसमें Orange Theme, Sliders, और GitHub Space Fix सब कुछ शामिल है।

In [ ]:
# @title 🔑 Step 1: GitHub & Engine Setup
import os, shutil

GITHUB_USER = "" # @param {type:"string"}
GITHUB_TOKEN = "" # @param {type:"string"}
REPO_NAME = "" # @param {type:"string"}

# Space Fix: अगर नाम में स्पेस है तो उसे हटाना
GITHUB_USER = GITHUB_USER.replace(" ", "")

if GITHUB_USER and GITHUB_TOKEN and REPO_NAME:
    REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git"
    if os.path.exists(REPO_NAME): shutil.rmtree(REPO_NAME)
    
    !git clone {REPO_URL}
    os.chdir(REPO_NAME)
    os.makedirs("outputs", exist_ok=True)
    
    !pip install -q gradio librosa soundfile coqui-tts
    print(f"✅ {REPO_NAME} तैयार है और यूजरनेम फिक्स हो गया है!")
else:
    print("⚠️ भाई, अपना डेटा सही से भरें!")

In [ ]:
# @title 🚀 Step 2: app.py (With All Tools & Orange Theme)
app_code = r'''
import gradio as gr
import torch
from TTS.api import TTS
import librosa, soundfile as sf
import os, re

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

def master_studio(text, audio_sample, speed, pitch, lang, sil_rem):
    if not audio_sample: return None
    text = text.replace('...', '. ')
    text = re.sub(r'([।?!,:;])', r' \1 ', text)
    
    out_path = 'outputs/v_batch_pro.wav'
    tts.tts_to_file(text=text, speaker_wav=audio_sample, language=lang, file_path=out_path, split_sentences=True)
    
    y, sr = librosa.load(out_path)
    if sil_rem: y, _ = librosa.effects.trim(y, top_db=25)
    if speed != 1.0: y = librosa.effects.time_stretch(y, rate=speed)
    if pitch != 0: y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch)
    
    sf.write(out_path, y, sr)
    return out_path

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown('# 🎙️ Professional Voice Studio v2.1.4')
    with gr.Row():
        with gr.Column():
            txt = gr.Textbox(label='Script (Tags: [laugh], [sigh])', lines=8)
            smp = gr.Audio(label='Voice Sample', type='filepath')
            lng = gr.Dropdown(choices=['hi', 'en', 'mr', 'bn'], label='Language', value='hi')
            with gr.Row():
                spd = gr.Slider(0.7, 1.3, 1.0, step=0.01, label="Speed Control")
                ptc = gr.Slider(-4, 4, 0, step=1, label="Pitch Control")
            sil = gr.Checkbox(label="Remove Extra Silence", value=True)
            btn = gr.Button('Generate Emotional Voice 🔱', variant='primary')
        with gr.Column():
            out = gr.Audio(label='Output Audio')
            gr.Markdown('**Expressions:** [laugh], [sigh], [cough], [clear_throat]')

    btn.click(master_studio, [txt, smp, spd, ptc, lng, sil], out)

demo.launch(share=True)
'''
with open('app.py', 'w') as f: f.write(app_code)
!python app.py

In [ ]:
# @title ⬆️ Step 3: GitHub Push (Safe Save)
!git config --global user.email "colab@example.com"
!git config --global user.name "{GITHUB_USER}"
!git add .
!git commit -m "Final Version with All Tools"
!git push
print("✅ सब कुछ GitHub पर सुरक्षित सेव हो गया है!")